In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
weather_data = "/Users/edwardamoah/Documents/GitHub/OsmiaPopModel/Python_scripts/notebooks/Nesting_Analysis/data/weather_data.txt"

In [4]:
"""
Parse weather data from tab-separated format to clean CSV.

Key techniques demonstrated:
1. Reading non-standard file formats with repeating headers
2. Using regex to extract numeric values from strings with units
3. Detecting date headers vs data rows
4. Handling European number formats (comma as thousands separator)
"""

import csv
import re
from datetime import datetime
from pathlib import Path


# Pattern to match date headers like "April 16, 2024"
DATE_PATTERN = re.compile(r"^[A-Z][a-z]+ \d{1,2}, \d{4}$")


def parse_date_to_iso(date_str: str) -> str:
    """
    Convert 'April 16, 2024' to '2024-04-16' (ISO format).
    
    ISO format sorts correctly as text and is unambiguous internationally.
    """
    dt = datetime.strptime(date_str, "%B %d, %Y")
    return dt.strftime("%Y-%m-%d")


def extract_numeric(value: str) -> str:
    """
    Extract numeric value from a string with units.
    
    Examples:
        "10.6 °C" -> "10.6"
        "1,008.74 hPa" -> "1008.74"
        "52 %" -> "52"
    """
    if not value.strip():
        return ""
    
    # Remove thousands separator (comma in numbers like 1,008.74)
    value = value.replace(",", "")
    
    # Extract the first number (including negative and decimal)
    match = re.search(r"-?\d+\.?\d*", value)
    return match.group(0) if match else value.strip()


def is_date_line(line: str) -> bool:
    """Check if a line is a date header."""
    return bool(DATE_PATTERN.match(line.strip()))


def is_header_line(line: str) -> bool:
    """Check if a line is the column headers."""
    return line.strip().startswith("Time\t")


def parse_weather_file(input_path: str, output_path: str) -> int:
    """
    Parse weather data file with multiple date sections.
    
    Structure per section:
      - Date line: "April 16, 2024"
      - Header line: "Time\tTemperature\t..."
      - Data rows until next date line
    
    Returns the number of data rows processed.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    # Output columns: Date + original headers
    output_headers = ["Date", "Time", "Temperature", "Dew Point", "Humidity", 
                      "Wind", "Speed", "Gust", "Pressure", "Precip. Rate", 
                      "Precip. Accum", "UV", "Solar"]
    
    rows = []
    current_date = None
    dates_found = []
    
    for line in lines:
        line_stripped = line.strip()
        
        # Skip empty lines
        if not line_stripped:
            continue
        
        # Check for date header
        if is_date_line(line_stripped):
            current_date = parse_date_to_iso(line_stripped)
            dates_found.append(current_date)
            continue
        
        # Skip column header lines
        if is_header_line(line_stripped):
            continue
        
        # This should be a data row
        if current_date is None:
            continue  # Skip orphan data before first date
            
        parts = line_stripped.split("\t")
        
        # Need at least Time column
        if len(parts) < 1:
            continue
        
        # Build row with date prepended
        row = {"Date": current_date}
        
        # Map parts to headers (skip "Date" which is index 0 in output_headers)
        data_headers = output_headers[1:]  # Time, Temperature, etc.
        
        for j, header in enumerate(data_headers):
            if j < len(parts):
                raw_value = parts[j]
                
                # Time and Wind stay as-is
                if header in ("Time", "Wind"):
                    row[header] = raw_value.strip()
                else:
                    row[header] = extract_numeric(raw_value)
            else:
                row[header] = ""
        
        rows.append(row)
    
    # Write CSV output
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=output_headers)
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"Found {len(dates_found)} days:")
    for d in dates_found[:5]:
        print(f"  - {d}")
    if len(dates_found) > 5:
        print(f"  ... and {len(dates_found) - 5} more")
    
    print(f"\nProcessed {len(rows)} data rows")
    print(f"Output written to: {output_path}")
    
    return len(rows)


if __name__ == "__main__":
    input_file = "/Users/edwardamoah/Documents/GitHub/OsmiaPopModel/Python_scripts/notebooks/Nesting_Analysis/data/weather_data.txt"
    output_file = "/Users/edwardamoah/Documents/GitHub/OsmiaPopModel/Python_scripts/notebooks/Nesting_Analysis/data/weather_data.csv"
    
    parse_weather_file(input_file, output_file)
    
    # Show first few rows of output
    print("\n--- First 5 rows of output ---")
    with open(output_file, "r") as f:
        for i, line in enumerate(f):
            if i < 6:
                print(line.strip())

Found 32 days:
  - 2024-04-16
  - 2024-04-17
  - 2024-04-18
  - 2024-04-19
  - 2024-04-20
  ... and 27 more

Processed 9210 data rows
Output written to: /Users/edwardamoah/Documents/GitHub/OsmiaPopModel/Python_scripts/notebooks/Nesting_Analysis/data/weather_data.csv

--- First 5 rows of output ---
Date,Time,Temperature,Dew Point,Humidity,Wind,Speed,Gust,Pressure,Precip. Rate,Precip. Accum,UV,Solar
2024-04-16,12:04 AM,10.6,1.2,52,East,0.0,0.0,1008.74,0.00,0.00,0,0
2024-04-16,12:09 AM,10.4,1.6,54,SE,0.0,0.0,1008.84,0.00,0.00,0,0
2024-04-16,12:14 AM,9.8,1.7,57,SE,0.0,0.0,1008.74,0.00,0.00,0,0
2024-04-16,12:19 AM,9.1,1.6,60,SSE,0.3,0.6,1008.94,0.00,0.00,0,0
2024-04-16,12:24 AM,8.7,1.8,62,ESE,0.1,0.3,1008.84,0.00,0.00,0,0


In [ ]:
"""

April 1, 2024 - April 30, 2024
Temperature	Dew Point	Humidity	Speed	Pressure	Precip. Accum.
Date	High	Avg	Low	High	Avg	Low	High	Avg	Low	High	Avg	Low	High	Low	Sum
4/1/2024	11.4 °C	7.6 °C	5.2 °C	9.6 °C	7.1 °C	4.7 °C	99 %	96 %	88 %	8.7 km/h	0.7 km/h	0.0 km/h	1,005.01 hPa	1,001.52 hPa	14.20 mm
4/2/2024	8.5 °C	7.6 °C	6.0 °C	8.4 °C	7.3 °C	5.8 °C	99 %	98 %	94 %	16.9 km/h	1.7 km/h	0.0 km/h	1,004.13 hPa	994.51 hPa	57.40 mm
4/3/2024	6.7 °C	5.5 °C	2.5 °C	6.6 °C	5.2 °C	2.3 °C	99 %	98 %	92 %	19.5 km/h	1.8 km/h	0.0 km/h	997.12 hPa	983.71 hPa	46.99 mm
4/4/2024	9.1 °C	3.1 °C	-1.8 °C	5.3 °C	1.7 °C	-2.0 °C	99 %	92 %	63 %	14.8 km/h	1.6 km/h	0.0 km/h	993.63 hPa	985.54 hPa	2.31 mm
4/5/2024	5.1 °C	3.2 °C	1.8 °C	2.4 °C	-0.7 °C	-3.4 °C	92 %	77 %	56 %	27.0 km/h	6.8 km/h	0.0 km/h	1,002.91 hPa	993.33 hPa	0.00 mm
4/6/2024	11.9 °C	6.8 °C	2.2 °C	2.9 °C	0.3 °C	-2.0 °C	84 %	65 %	43 %	33.5 km/h	7.6 km/h	0.0 km/h	1,007.82 hPa	1,002.24 hPa	0.00 mm
4/7/2024	16.8 °C	8.7 °C	-0.5 °C	2.1 °C	-0.8 °C	-2.9 °C	92 %	55 %	27 %	21.9 km/h	2.9 km/h	0.0 km/h	1,011.31 hPa	1,007.52 hPa	0.00 mm
4/8/2024	14.4 °C	6.6 °C	-1.1 °C	8.0 °C	1.6 °C	-2.7 °C	90 %	72 %	46 %	5.0 km/h	0.4 km/h	0.0 km/h	1,012.63 hPa	1,008.33 hPa	0.20 mm
4/9/2024	25.1 °C	12.9 °C	-0.1 °C	12.8 °C	6.9 °C	-0.3 °C	99 %	72 %	37 %	16.3 km/h	1.5 km/h	0.0 km/h	1,009.82 hPa	1,003.62 hPa	0.00 mm
4/10/2024	19.9 °C	16.3 °C	13.1 °C	14.8 °C	13.1 °C	11.1 °C	94 %	82 %	69 %	7.6 km/h	0.7 km/h	0.0 km/h	1,006.74 hPa	1,004.13 hPa	0.00 mm
4/11/2024	20.3 °C	16.4 °C	12.9 °C	17.0 °C	14.8 °C	12.7 °C	99 %	91 %	72 %	24.8 km/h	3.8 km/h	0.0 km/h	1,004.84 hPa	984.32 hPa	10.21 mm
4/12/2024	18.5 °C	11.8 °C	6.1 °C	16.9 °C	9.8 °C	3.9 °C	99 %	88 %	71 %	43.1 km/h	6.0 km/h	0.0 km/h	991.94 hPa	979.41 hPa	33.81 mm
4/13/2024	13.0 °C	8.2 °C	4.6 °C	4.7 °C	2.1 °C	0.4 °C	94 %	67 %	48 %	41.4 km/h	9.3 km/h	0.0 km/h	1,005.42 hPa	991.64 hPa	3.51 mm
4/14/2024	27.5 °C	13.4 °C	2.9 °C	13.6 °C	5.9 °C	0.1 °C	99 %	64 %	34 %	30.6 km/h	3.1 km/h	0.0 km/h	1,005.22 hPa	992.62 hPa	14.30 mm
4/15/2024	23.8 °C	16.9 °C	9.4 °C	14.8 °C	9.3 °C	0.2 °C	99 %	67 %	31 %	29.5 km/h	4.3 km/h	0.0 km/h	1,008.74 hPa	997.83 hPa	0.00 mm
4/16/2024	24.8 °C	13.7 °C	2.0 °C	9.2 °C	3.9 °C	0.2 °C	90 %	56 %	25 %	15.4 km/h	1.1 km/h	0.0 km/h	1,013.24 hPa	1,008.13 hPa	0.00 mm
4/17/2024	18.7 °C	12.8 °C	7.5 °C	14.0 °C	8.9 °C	4.0 °C	99 %	77 %	60 %	13.0 km/h	0.6 km/h	0.0 km/h	1,009.62 hPa	1,001.52 hPa	9.91 mm
4/18/2024	19.9 °C	15.1 °C	9.4 °C	14.1 °C	11.8 °C	8.3 °C	99 %	82 %	55 %	19.5 km/h	2.1 km/h	0.0 km/h	1,008.84 hPa	1,002.34 hPa	2.31 mm
4/19/2024	12.4 °C	10.4 °C	7.4 °C	11.0 °C	9.6 °C	7.2 °C	99 %	95 %	88 %	13.0 km/h	1.9 km/h	0.0 km/h	1,008.43 hPa	1,004.64 hPa	0.00 mm
4/20/2024	16.4 °C	10.7 °C	3.2 °C	10.1 °C	1.8 °C	-5.0 °C	99 %	57 %	33 %	34.6 km/h	5.8 km/h	0.0 km/h	1,009.31 hPa	1,003.73 hPa	0.00 mm
4/21/2024	8.2 °C	3.9 °C	-1.7 °C	-0.1 °C	-2.6 °C	-5.4 °C	88 %	64 %	50 %	27.0 km/h	3.1 km/h	0.0 km/h	1,009.21 hPa	1,005.22 hPa	0.00 mm
4/22/2024	16.4 °C	8.0 °C	-2.5 °C	1.6 °C	-1.8 °C	-4.4 °C	89 %	54 %	27 %	27.7 km/h	3.3 km/h	0.0 km/h	1,010.33 hPa	1,007.32 hPa	0.00 mm
4/23/2024	21.1 °C	11.3 °C	-0.1 °C	5.2 °C	1.3 °C	-1.6 °C	91 %	55 %	27 %	20.6 km/h	2.8 km/h	0.0 km/h	1,010.63 hPa	1,000.44 hPa	0.00 mm
4/24/2024	17.1 °C	12.5 °C	7.5 °C	10.1 °C	6.9 °C	1.3 °C	91 %	70 %	44 %	27.0 km/h	5.1 km/h	0.0 km/h	1,011.92 hPa	1,000.03 hPa	0.71 mm
4/25/2024	14.8 °C	7.8 °C	2.9 °C	4.1 °C	-1.0 °C	-5.2 °C	85 %	56 %	29 %	12.2 km/h	2.1 km/h	0.0 km/h	1,017.81 hPa	1,011.72 hPa	0.00 mm
4/26/2024	18.8 °C	10.3 °C	-0.5 °C	7.6 °C	1.0 °C	-4.6 °C	75 %	53 %	42 %	19.8 km/h	3.1 km/h	0.0 km/h	1,019.74 hPa	1,016.02 hPa	0.00 mm
4/27/2024	11.9 °C	9.6 °C	7.8 °C	9.5 °C	5.4 °C	0.6 °C	99 %	77 %	48 %	16.3 km/h	2.0 km/h	0.0 km/h	1,019.81 hPa	1,014.22 hPa	2.31 mm
4/28/2024	28.9 °C	18.3 °C	9.6 °C	18.7 °C	13.5 °C	9.4 °C	99 %	77 %	44 %	22.4 km/h	1.4 km/h	0.0 km/h	1,014.63 hPa	1,007.42 hPa	0.30 mm
4/29/2024	30.1 °C	21.5 °C	11.6 °C	18.6 °C	15.1 °C	11.4 °C	99 %	71 %	42 %	16.3 km/h	1.3 km/h	0.0 km/h	1,008.64 hPa	1,002.54 hPa	0.00 mm
4/30/2024	28.6 °C	19.2 °C	13.3 °C	18.2 °C	15.3 °C	12.8 °C	99 %	81 %	45 %	27.4 km/h	1.8 km/h	0.0 km/h	1,004.03 hPa	1,000.03 hPa	1.70 mm








May 1, 2024 - May 31, 2024
Temperature	Dew Point	Humidity	Speed	Pressure	Precip. Accum.
Date	High	Avg	Low	High	Avg	Low	High	Avg	Low	High	Avg	Low	High	Low	Sum
5/1/2024	27.4 °C	19.0 °C	9.6 °C	14.3 °C	11.3 °C	9.1 °C	99 %	67 %	33 %	14.8 km/h	1.6 km/h	0.0 km/h	1,005.83 hPa	1,002.54 hPa	0.30 mm
5/2/2024	30.5 °C	21.0 °C	11.5 °C	14.3 °C	9.4 °C	4.7 °C	87 %	53 %	20 %	23.0 km/h	2.3 km/h	0.0 km/h	1,007.11 hPa	1,003.52 hPa	0.00 mm
5/3/2024	25.0 °C	17.9 °C	8.8 °C	15.6 °C	11.5 °C	7.5 °C	93 %	68 %	46 %	20.1 km/h	1.7 km/h	0.0 km/h	1,010.94 hPa	1,006.64 hPa	0.00 mm
5/4/2024	15.5 °C	11.0 °C	9.7 °C	13.5 °C	10.4 °C	9.1 °C	99 %	96 %	86 %	11.6 km/h	1.1 km/h	0.0 km/h	1,013.72 hPa	1,010.23 hPa	11.40 mm
5/5/2024	14.7 °C	11.4 °C	8.7 °C	13.6 °C	10.9 °C	8.5 °C	99 %	97 %	90 %	11.1 km/h	0.8 km/h	0.0 km/h	1,012.02 hPa	1,008.43 hPa	6.60 mm
5/6/2024	25.2 °C	17.1 °C	12.2 °C	19.2 °C	15.2 °C	12.0 °C	99 %	89 %	60 %	10.8 km/h	0.3 km/h	0.0 km/h	1,008.94 hPa	1,002.13 hPa	0.30 mm
5/7/2024	25.4 °C	18.5 °C	13.8 °C	19.5 °C	15.4 °C	10.0 °C	99 %	83 %	63 %	11.6 km/h	1.0 km/h	0.0 km/h	1,002.54 hPa	995.94 hPa	0.00 mm
5/8/2024	27.5 °C	21.8 °C	16.2 °C	18.7 °C	13.2 °C	3.1 °C	99 %	64 %	29 %	30.3 km/h	4.6 km/h	0.0 km/h	998.14 hPa	993.84 hPa	2.31 mm
5/9/2024	18.5 °C	13.3 °C	9.5 °C	11.9 °C	10.0 °C	5.2 °C	99 %	82 %	42 %	13.7 km/h	1.1 km/h	0.0 km/h	999.63 hPa	996.51 hPa	13.89 mm
5/10/2024	10.7 °C	9.6 °C	8.7 °C	10.6 °C	9.4 °C	8.5 °C	99 %	99 %	99 %	10.5 km/h	0.8 km/h	0.0 km/h	1,001.42 hPa	995.23 hPa	28.19 mm
5/11/2024	16.4 °C	10.4 °C	7.9 °C	10.7 °C	8.8 °C	7.5 °C	99 %	91 %	58 %	15.4 km/h	1.6 km/h	0.0 km/h	1,001.63 hPa	997.43 hPa	8.71 mm
5/12/2024	16.5 °C	11.4 °C	7.6 °C	12.5 °C	9.6 °C	7.0 °C	99 %	90 %	66 %	21.6 km/h	1.8 km/h	0.0 km/h	1,006.33 hPa	997.63 hPa	3.51 mm
5/13/2024	25.4 °C	15.2 °C	4.5 °C	14.7 °C	10.0 °C	4.3 °C	99 %	75 %	46 %	19.5 km/h	2.3 km/h	0.0 km/h	1,008.13 hPa	1,003.52 hPa	0.00 mm
5/14/2024	18.5 °C	14.3 °C	8.8 °C	15.6 °C	13.0 °C	8.6 °C	99 %	92 %	79 %	14.3 km/h	0.7 km/h	0.0 km/h	1,005.52 hPa	1,000.71 hPa	2.79 mm
5/15/2024	18.9 °C	15.3 °C	12.6 °C	16.2 °C	14.5 °C	12.4 °C	99 %	95 %	81 %	11.1 km/h	0.4 km/h	0.0 km/h	1,001.02 hPa	998.14 hPa	2.11 mm
5/16/2024	25.2 °C	17.2 °C	12.1 °C	17.5 °C	14.2 °C	11.9 °C	99 %	85 %	54 %	12.2 km/h	0.7 km/h	0.0 km/h	1,002.91 hPa	998.82 hPa	0.00 mm
5/17/2024	21.3 °C	16.3 °C	12.4 °C	16.3 °C	14.3 °C	12.2 °C	99 %	89 %	71 %	14.8 km/h	1.5 km/h	0.0 km/h	1,005.83 hPa	1,002.54 hPa	0.20 mm
5/18/2024	22.8 °C	17.7 °C	15.3 °C	18.3 °C	16.2 °C	15.1 °C	99 %	92 %	72 %	17.2 km/h	1.4 km/h	0.0 km/h	1,005.52 hPa	1,003.01 hPa	0.79 mm
5/19/2024	25.3 °C	18.8 °C	13.9 °C	19.2 °C	16.0 °C	13.7 °C	99 %	85 %	61 %	10.5 km/h	0.7 km/h	0.0 km/h	1,007.72 hPa	1,005.01 hPa	0.00 mm
5/20/2024	29.1 °C	20.2 °C	11.2 °C	19.8 °C	15.7 °C	11.0 °C	99 %	79 %	49 %	12.2 km/h	1.2 km/h	0.0 km/h	1,008.74 hPa	1,004.84 hPa	0.00 mm
5/21/2024	30.0 °C	22.2 °C	13.8 °C	18.7 °C	16.1 °C	13.6 °C	99 %	72 %	39 %	19.5 km/h	2.2 km/h	0.0 km/h	1,007.11 hPa	1,003.52 hPa	0.00 mm
5/22/2024	30.3 °C	21.5 °C	14.3 °C	22.4 °C	18.7 °C	14.1 °C	99 %	85 %	53 %	24.8 km/h	1.4 km/h	0.0 km/h	1,005.83 hPa	1,001.52 hPa	9.91 mm
5/23/2024	26.9 °C	21.1 °C	15.0 °C	21.0 °C	16.6 °C	12.8 °C	99 %	78 %	45 %	13.7 km/h	1.4 km/h	0.0 km/h	1,005.42 hPa	1,001.42 hPa	19.99 mm
5/24/2024	29.0 °C	20.2 °C	11.0 °C	16.6 °C	13.2 °C	10.8 °C	99 %	69 %	36 %	17.2 km/h	1.5 km/h	0.0 km/h	1,004.84 hPa	1,000.54 hPa	0.00 mm
5/25/2024	29.1 °C	18.7 °C	10.8 °C	22.0 °C	15.8 °C	10.6 °C	99 %	85 %	53 %	17.2 km/h	1.3 km/h	0.0 km/h	1,003.73 hPa	999.83 hPa	11.99 mm
5/26/2024	27.6 °C	19.6 °C	15.2 °C	20.9 °C	16.9 °C	15.0 °C	99 %	86 %	59 %	13.0 km/h	1.0 km/h	0.0 km/h	1,006.23 hPa	998.92 hPa	0.20 mm
5/27/2024	26.9 °C	20.4 °C	15.3 °C	21.6 °C	18.0 °C	15.1 °C	99 %	87 %	60 %	17.2 km/h	1.7 km/h	0.0 km/h	1,000.61 hPa	994.04 hPa	8.41 mm
5/28/2024	22.8 °C	17.8 °C	13.9 °C	17.4 °C	13.6 °C	12.2 °C	96 %	77 %	61 %	21.2 km/h	3.4 km/h	0.0 km/h	1,004.64 hPa	998.14 hPa	5.31 mm
5/29/2024	22.2 °C	14.1 °C	9.2 °C	16.0 °C	12.3 °C	9.0 °C	99 %	90 %	62 %	13.4 km/h	1.3 km/h	0.0 km/h	1,006.94 hPa	1,003.62 hPa	26.70 mm
5/30/2024	20.6 °C	14.8 °C	9.0 °C	12.1 °C	8.1 °C	4.2 °C	99 %	68 %	39 %	22.7 km/h	3.6 km/h	0.0 km/h	1,011.92 hPa	1,006.74 hPa	0.00 mm
5/31/2024	23.7 °C	14.6 °C	3.8 °C	9.6 °C	6.3 °C	3.5 °C	99 %	63 %	31 %	18.0 km/h	2.0 km/h	0.0 km/h	1,016.53 hPa






July 1, 2024 - July 31, 2024
Temperature	Dew Point	Humidity	Speed	Pressure	Precip. Accum.
Date	High	Avg	Low	High	Avg	Low	High	Avg	Low	High	Avg	Low	High	Low	Sum
7/1/2024	24.2 °C	18.0 °C	13.3 °C	15.5 °C	12.6 °C	10.9 °C	96 %	72 %	46 %	19.2 km/h	2.5 km/h	0.0 km/h	1,014.53 hPa	1,009.72 hPa	0.00 mm
7/2/2024	27.2 °C	18.7 °C	9.5 °C	18.3 °C	14.2 °C	9.3 °C	99 %	78 %	49 %	16.3 km/h	1.6 km/h	0.0 km/h	1,017.03 hPa	1,011.72 hPa	0.00 mm
7/3/2024	29.5 °C	22.9 °C	16.0 °C	22.1 °C	18.3 °C	14.5 °C	93 %	76 %	60 %	19.5 km/h	2.9 km/h	0.0 km/h	1,012.84 hPa	1,006.23 hPa	0.00 mm
7/4/2024	32.4 °C	25.4 °C	21.5 °C	25.5 °C	22.5 °C	19.7 °C	99 %	85 %	62 %	14.3 km/h	1.0 km/h	0.0 km/h	1,007.32 hPa	1,001.63 hPa	3.81 mm
7/5/2024	31.9 °C	25.2 °C	21.2 °C	26.2 °C	23.4 °C	21.0 °C	99 %	90 %	67 %	9.7 km/h	0.9 km/h	0.0 km/h	1,002.81 hPa	999.42 hPa	2.01 mm
7/6/2024	32.0 °C	26.7 °C	21.9 °C	23.8 °C	21.2 °C	17.7 °C	98 %	74 %	45 %	16.3 km/h	2.0 km/h	0.0 km/h	1,006.13 hPa	1,000.03 hPa	0.00 mm
7/7/2024	31.6 °C	23.9 °C	16.4 °C	22.6 °C	18.9 °C	16.2 °C	99 %	76 %	46 %	10.5 km/h	0.9 km/h	0.0 km/h	1,009.21 hPa	1,005.93 hPa	0.00 mm
7/8/2024	32.9 °C	24.5 °C	17.3 °C	25.0 °C	20.7 °C	17.1 °C	99 %	82 %	52 %	18.0 km/h	0.9 km/h	0.0 km/h	1,009.62 hPa	1,006.03 hPa	0.00 mm
7/9/2024	33.3 °C	26.5 °C	18.9 °C	24.9 °C	22.0 °C	18.7 °C	99 %	79 %	49 %	12.6 km/h	0.2 km/h	0.0 km/h	1,007.42 hPa	1,003.32 hPa	0.00 mm
7/10/2024	34.1 °C	26.3 °C	20.9 °C	27.2 °C	23.2 °C	18.0 °C	99 %	85 %	61 %	33.8 km/h	3.7 km/h	0.0 km/h	1,003.73 hPa	994.31 hPa	23.39 mm
7/11/2024	25.0 °C	22.1 °C	20.1 °C	21.1 °C	19.0 °C	17.3 °C	93 %	82 %	74 %	18.0 km/h	2.6 km/h	0.0 km/h	1,010.74 hPa	1,000.54 hPa	0.00 mm
7/12/2024	31.6 °C	24.0 °C	17.0 °C	22.0 °C	19.8 °C	16.8 °C	99 %	80 %	48 %	15.1 km/h	0.9 km/h	0.0 km/h	1,013.04 hPa	1,010.13 hPa	0.00 mm
7/13/2024	33.8 °C	24.9 °C	17.1 °C	25.2 °C	20.9 °C	16.9 °C	99 %	81 %	50 %	11.1 km/h	0.5 km/h	0.0 km/h	1,013.72 hPa	1,009.21 hPa	0.00 mm
7/14/2024	34.4 °C	25.9 °C	18.6 °C	24.1 °C	20.1 °C	16.1 °C	99 %	75 %	36 %	14.8 km/h	1.1 km/h	0.0 km/h	1,010.94 hPa	1,004.54 hPa	0.00 mm
7/15/2024	34.9 °C	27.7 °C	21.3 °C	25.5 °C	21.0 °C	18.6 °C	92 %	69 %	40 %	21.2 km/h	2.1 km/h	0.0 km/h	1,005.93 hPa	999.02 hPa	0.00 mm
7/16/2024	34.1 °C	27.0 °C	19.4 °C	24.4 °C	21.0 °C	19.0 °C	98 %	72 %	46 %	23.3 km/h	2.4 km/h	0.0 km/h	1,002.13 hPa	999.42 hPa	0.00 mm
7/17/2024	28.5 °C	22.6 °C	19.3 °C	24.1 °C	20.9 °C	18.9 °C	99 %	91 %	71 %	22.4 km/h	1.2 km/h	0.0 km/h	1,004.13 hPa	1,001.02 hPa	0.79 mm
7/18/2024	27.3 °C	21.9 °C	16.6 °C	19.9 °C	17.3 °C	13.1 °C	99 %	78 %	50 %	18.7 km/h	1.7 km/h	0.0 km/h	1,010.02 hPa	1,003.42 hPa	0.00 mm
7/19/2024	28.1 °C	19.9 °C	11.6 °C	18.0 °C	14.5 °C	11.4 °C	99 %	74 %	45 %	12.6 km/h	0.8 km/h	0.0 km/h	1,013.51 hPa	1,009.41 hPa	0.00 mm
7/20/2024	29.2 °C	21.4 °C	15.1 °C	22.7 °C	18.4 °C	14.7 °C	99 %	84 %	60 %	14.8 km/h	1.2 km/h	0.0 km/h	1,011.41 hPa	1,007.11 hPa	0.00 mm
7/21/2024	29.7 °C	22.2 °C	16.2 °C	20.6 °C	17.7 °C	14.9 °C	99 %	79 %	43 %	10.5 km/h	0.7 km/h	0.0 km/h	1,010.02 hPa	1,007.62 hPa	0.00 mm
7/22/2024	28.2 °C	20.2 °C	15.5 °C	22.9 °C	18.6 °C	15.3 °C	99 %	91 %	67 %	23.0 km/h	0.8 km/h	0.0 km/h	1,009.82 hPa	1,005.11 hPa	34.49 mm
7/23/2024	28.4 °C	22.5 °C	17.4 °C	21.6 °C	19.5 °C	17.2 °C	99 %	85 %	59 %	9.3 km/h	0.8 km/h	0.0 km/h	1,010.43 hPa	1,005.01 hPa	6.91 mm
7/24/2024	29.8 °C	22.8 °C	18.5 °C	23.2 °C	20.3 °C	18.3 °C	99 %	87 %	61 %	22.4 km/h	1.2 km/h	0.0 km/h	1,010.84 hPa	1,007.62 hPa	1.80 mm
7/25/2024	27.6 °C	22.0 °C	17.6 °C	20.2 °C	17.7 °C	14.0 °C	99 %	79 %	51 %	16.9 km/h	1.7 km/h	0.0 km/h	1,012.33 hPa	1,009.04 hPa	0.20 mm
7/26/2024	27.4 °C	19.6 °C	11.4 °C	18.5 °C	14.7 °C	11.2 °C	99 %	76 %	45 %	6.9 km/h	0.6 km/h	0.0 km/h	1,014.22 hPa	1,010.33 hPa	0.00 mm
7/27/2024	29.5 °C	20.0 °C	11.6 °C	19.5 °C	15.6 °C	11.4 °C	99 %	79 %	45 %	11.6 km/h	0.5 km/h	0.0 km/h	1,014.02 hPa	1,010.63 hPa	0.00 mm
7/28/2024	30.9 °C	21.1 °C	11.9 °C	21.4 °C	16.5 °C	11.7 °C	99 %	78 %	45 %	7.9 km/h	0.3 km/h	0.0 km/h	1,013.34 hPa	1,007.62 hPa	0.00 mm
7/29/2024	33.2 °C	23.7 °C	14.2 °C	22.6 °C	18.4 °C	14.0 °C	99 %	76 %	40 %	14.3 km/h	0.4 km/h	0.0 km/h	1,008.43 hPa	1,002.91 hPa	0.00 mm
7/30/2024	29.7 °C	22.9 °C	19.5 °C	24.9 °C	21.5 °C	19.3 °C	99 %	92 %	69 %	17.2 km/h	1.3 km/h	0.0 km/h	1,004.84 hPa	1,000.91 hPa	16.79 mm
7/31/2024	31.5 °C	25.2 °C	20.4 °C	24.2 °C	21.8 °C	20.2 °C	99 %	83 %	60 %	15.8 km/h	1.7 km/h	0.0 km/h	1,006.54 hPa	1,002.03 hPa	2.31 mm






"""

'\n\nApril 1, 2024 - April 30, 2024\nTemperature\tDew Point\tHumidity\tSpeed\tPressure\tPrecip. Accum.\nDate\tHigh\tAvg\tLow\tHigh\tAvg\tLow\tHigh\tAvg\tLow\tHigh\tAvg\tLow\tHigh\tLow\tSum\n4/1/2024\t11.4 °C\t7.6 °C\t5.2 °C\t9.6 °C\t7.1 °C\t4.7 °C\t99 %\t96 %\t88 %\t8.7 km/h\t0.7 km/h\t0.0 km/h\t1,005.01 hPa\t1,001.52 hPa\t14.20 mm\n4/2/2024\t8.5 °C\t7.6 °C\t6.0 °C\t8.4 °C\t7.3 °C\t5.8 °C\t99 %\t98 %\t94 %\t16.9 km/h\t1.7 km/h\t0.0 km/h\t1,004.13 hPa\t994.51 hPa\t57.40 mm\n4/3/2024\t6.7 °C\t5.5 °C\t2.5 °C\t6.6 °C\t5.2 °C\t2.3 °C\t99 %\t98 %\t92 %\t19.5 km/h\t1.8 km/h\t0.0 km/h\t997.12 hPa\t983.71 hPa\t46.99 mm\n4/4/2024\t9.1 °C\t3.1 °C\t-1.8 °C\t5.3 °C\t1.7 °C\t-2.0 °C\t99 %\t92 %\t63 %\t14.8 km/h\t1.6 km/h\t0.0 km/h\t993.63 hPa\t985.54 hPa\t2.31 mm\n4/5/2024\t5.1 °C\t3.2 °C\t1.8 °C\t2.4 °C\t-0.7 °C\t-3.4 °C\t92 %\t77 %\t56 %\t27.0 km/h\t6.8 km/h\t0.0 km/h\t1,002.91 hPa\t993.33 hPa\t0.00 mm\n4/6/2024\t11.9 °C\t6.8 °C\t2.2 °C\t2.9 °C\t0.3 °C\t-2.0 °C\t84 %\t65 %\t43 %\t33.5 km/h\t7.6 km